In [1]:

from pyspark.sql import functions as F

# 1. Read the combined Bronze table
df_bronze = spark.table("bronze_online_retail_all")

print("Bronze rows:", f"{df_bronze.count():,}")
print("Source columns:", df_bronze.columns)

# 2. Verify the expected columns exist
required_columns = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "price",
    "customer_id",
    "country",
]

missing_columns = [
    col_name for col_name in required_columns
    if col_name not in df_bronze.columns
]

if missing_columns:
    raise ValueError(
        f"Missing expected columns: {missing_columns}. "
        f"Available columns: {df_bronze.columns}"
    )

# 3. Standardize text and convert data types
df_silver = (
    df_bronze
    .withColumn("invoice", F.trim(F.col("invoice")))
    .withColumn("stock_code", F.trim(F.col("stock_code")))
    .withColumn("description", F.trim(F.col("description")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id")).cast("string")
    )
    .withColumn(
        "quantity",
        F.col("quantity").cast("integer")
    )
    .withColumn(
        "price",
        F.col("price").cast("decimal(18,4)")
    )
    .withColumn(
        "invoice_date",
        F.coalesce(
            F.to_timestamp("invoice_date", "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp("invoice_date", "M/d/yyyy H:mm"),
            F.to_timestamp("invoice_date", "MM/dd/yyyy HH:mm"),
            F.to_timestamp("invoice_date")
        )
    )
)

# 4. Add analytical columns and data-quality flags
df_silver = (
    df_silver
    .withColumn(
        "sales_amount",
        F.when(
            F.col("quantity").isNotNull()
            & F.col("price").isNotNull(),
            F.col("quantity").cast("decimal(18,4)")
            * F.col("price")
        ).otherwise(F.lit(None).cast("decimal(38,8)"))
    )
    .withColumn(
        "is_cancellation_invoice",
        F.upper(F.col("invoice")).startswith("C")
    )
    .withColumn(
        "is_negative_quantity",
        F.col("quantity") < 0
    )
    .withColumn(
        "is_missing_customer",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "has_invalid_core_fields",
        F.col("invoice").isNull()
        | (F.col("invoice") == "")
        | F.col("invoice_date").isNull()
        | F.col("quantity").isNull()
        | F.col("price").isNull()
    )
)

# 5. Write the Silver Delta table
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_online_retail")
)

print("Silver table created: silver_online_retail")
print("Silver rows:", f"{df_silver.count():,}")

# 6. Display sample records
display(
    spark.table("silver_online_retail").limit(10)
)


StatementMeta(, 9d14f1ec-4c27-4757-97e8-0d678d638d52, 3, Finished, Available, Finished, False)

Bronze rows: 1,067,371
Source columns: ['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country', '_source_file', '_ingested_at']
Silver table created: silver_online_retail
Silver rows: 1,067,371


SynapseWidget(Synapse.DataFrame, 80f8a8f2-cdd4-4732-b93f-250c8bbeea73)

In [2]:

df = spark.table("silver_online_retail")

print("=== SILVER DATA QUALITY REPORT ===")
print("Total rows:", f"{df.count():,}")

print(
    "Invalid core-field rows:",
    f"{df.filter(F.col('has_invalid_core_fields')).count():,}"
)

print(
    "Cancellation invoice rows:",
    f"{df.filter(F.col('is_cancellation_invoice')).count():,}"
)

print(
    "Negative quantity rows:",
    f"{df.filter(F.col('is_negative_quantity')).count():,}"
)

print(
    "Rows with missing customer:",
    f"{df.filter(F.col('is_missing_customer')).count():,}"
)

print(
    "Rows with missing invoice date:",
    f"{df.filter(F.col('invoice_date').isNull()).count():,}"
)

print("\n=== COUNTRY SUMMARY ===")

country_summary = (
    df.groupBy("country")
    .agg(
        F.count("*").alias("transaction_rows"),
        F.sum("quantity").alias("total_units"),
        F.round(F.sum("sales_amount"), 2).alias("sales_amount")
    )
    .orderBy(F.desc("sales_amount"))
)

display(country_summary.limit(10))


StatementMeta(, 9d14f1ec-4c27-4757-97e8-0d678d638d52, 4, Finished, Available, Finished, False)

=== SILVER DATA QUALITY REPORT ===
Total rows: 1,067,371
Invalid core-field rows: 0
Cancellation invoice rows: 19,494
Negative quantity rows: 22,950
Rows with missing customer: 243,007
Rows with missing invoice date: 0

=== COUNTRY SUMMARY ===


SynapseWidget(Synapse.DataFrame, ef61f235-60a2-4a81-9eb2-9ad65532b7ad)